# U2-L1：ROS2 通信基础

**课程目标**

1. 理解为什么机器人系统需要 ROS2——多进程通信需求；认识 Node（节点）、Topic（话题）、发布-订阅、Message（消息）等基础通信概念（ROS2 通信五元组）。
2. 掌握 ROS2 话题观测四件套（`ros2 topic list` / `echo` / `hz` / `info`），能读懂 K1 当前系统中的话题与消息类型。
3. 通过一次简单的发布/订阅体验，理解「发布者与订阅者按话题名解耦」这一 ROS2 核心机制。


## 一、ROS2 系统观察：话题观测四件套

**目标**：在本机（或 K1）已加载 ROS2 环境的终端中，用四件套命令观察机器人当前的话题通信状况，建立「ROS2 不是操作系统、而是机器人软件之间的通信框架」的直观认识（核心概念 C01 节点 / C02 话题 / C05 消息）。

⚠️ **安全**：本实验为**观测课**，仅读取系统信息，**不发送任何运动 / 动作指令**；若误调用控制服务导致机器人移动，立即按下背部 `STAND` 按钮回到准备模式。运行前确认 ROS2 环境已加载（Booster Studio 已预置；独立环境需先 `source /opt/ros/humble/setup.bash`）。多组共用机器人时，话题命名加组号后缀避免互相干扰。

📌 **前置**：已完成 U1 四课（连接、模式、运动、头/步态/动作）；本实验不依赖 `BoosterRobot()` SDK，而是直接操作 ROS2 命令行。


### 1.1 确认 ROS2 环境

运行下面单元格确认 `ros2` 命令可用；若报 `command not found`，请先在终端执行 `source /opt/ros/humble/setup.bash` 后再启动本 notebook 的 kernel。


In [ ]:
!ros2 --version

### 1.2 列出当前话题（list）

`ros2 topic list` 列出系统中所有正在广播的话题。每一行是一个**话题（Topic）**——一条持续的数据通道。


In [ ]:
!ros2 topic list

### 1.3 查看单个话题详情（info）

`ros2 topic info <话题名>` 显示该话题的**消息类型（Type）**、发布者数量（Publisher count）与订阅者数量（Subscription count）。下面以图像话题为例；若本机话题名不同，先用 `ros2 topic list` 找到真实图像话题名再替换。


In [ ]:
!ros2 topic info /camera/image_raw    # 若本话题不存在，先 ros2 topic list 查看真实图像话题名（如 /boostercamera/head/rgb）再替换

### 1.4 订阅并查看一条消息（echo）

`ros2 topic echo <话题名>` 实时打印该话题发布的消息；`--once` 只打印最新一条后退出，适合快速查看消息结构。


In [ ]:
!ros2 topic echo /camera/image_raw --once

### 1.5 测量发布频率（hz）

`ros2 topic hz <话题名>` 测量话题的发布频率（Hz）。运行后会持续统计，请手动中断（菜单「内核 → 中断」或按 I/I）后继续。


In [ ]:
!ros2 topic hz /camera/image_raw

## 二、发布与订阅体验（五元组的 pub / sub）

**目标**：亲手发布一条字符串消息到自建话题，再用 `echo` 订阅观察，体会「发布者（C03）与订阅者（C04）不需要互相知道对方、只约定同一话题名与消息类型」的解耦本质（轻量概念 E01 事件循环 / E02 回调的雏形）。

<font color="red">本实验仅向自建话题 `/greeting_u2l1` 发布一条字符串，**不控制机器人**；若多组共用环境，请把话题名改为 `/greeting_u2l1_groupN`（N 为组号）以免互相干扰。</font>


### 2.1 启动发布者（后台进程）

下面用 Python `subprocess` 在后台启动一个 `ros2 topic pub`，每秒发布一次问候消息；随后在 2.2/2.3 用 `echo` 观察，最后在 2.4 自动停止发布者。


In [ ]:
import subprocess, time, signal, os

TOPIC = "/greeting_u2l1"   # 多组请改为 /greeting_u2l1_groupN
pub = subprocess.Popen(
    ["ros2", "topic", "pub", "-r", "2", TOPIC, "std_msgs/String", "data: Hello K1"],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE,
)
print("发布者已启动，PID =", pub.pid, " 话题 =", TOPIC)
time.sleep(2)

### 2.2 订阅并查看（echo / info）

发布者已在后台运行。下面查看该话题的发布/订阅数量，并读取一条消息。


In [ ]:
!ros2 topic info /greeting_u2l1
print("--- 读取一条消息 ---")
!ros2 topic echo /greeting_u2l1 --once

### 2.3 体验「多订阅者」

ROS2 允许同一话题有多个订阅者。再启动一个 `echo`（后台），然后查看 `info` 中 Subscription count 是否从 1 变为 2——这正是「发布者不感知订阅者」的解耦体现。


In [ ]:
sub = subprocess.Popen(
    ["ros2", "topic", "echo", "/greeting_u2l1"],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE,
)
print("第二个订阅者已启动，PID =", sub.pid)
time.sleep(2)
!ros2 topic info /greeting_u2l1
sub.send_signal(signal.SIGINT); sub.wait()
print("第二个订阅者已停止")

### 2.4 停止发布者并收尾


In [ ]:
pub.send_signal(signal.SIGINT); pub.wait()
print("发布者已停止，话题 /greeting_u2l1 已无发布者")

## 三、观察与记录（非代码）

| 观察项 | 你看到的输出 | 说明 |
| --- | --- | --- |
| `ros2 topic list` 话题数量 |  | 系统里大概有几个数据通道？ |
| 图像话题的 Type |  | 是 `sensor_msgs/msg/Image` 吗？ |
| 图像话题的 Pub / Sub 数 |  | 谁在发？谁在看？ |
| `hz` 测得的帧率 |  | 大致多少 Hz？ |
| 发布/订阅实验中 Sub count 的变化 |  | 多一个订阅者为啥发布者无感？ |

> 想一想：为什么 `ros2 topic pub` 和 `ros2 topic echo` 两个互不相识的终端，能通过 `/greeting_u2l1` 这个名字交换消息？如果把话题名改错一个字母会怎样？
